# Lesson 12 — Virtual Paint App

In [1]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)


In [2]:
# Blue color range
lower = np.array([100, 150, 0])
upper = np.array([140, 255, 255])


In [3]:
points = []   # stores center points of the detected object


In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break


In [ ]:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower, upper)


In [ ]:
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    center = None
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        if area > 500:
            (x, y, w, h) = cv2.boundingRect(cnt)
            center = (x + w//2, y + h//2)
            points.append(center)


In [ ]:
    for i in range(1, len(points)):
        if points[i-1] is None or points[i] is None:
            continue
        cv2.line(frame, points[i-1], points[i], (255,0,0), 5)  # blue line


In [ ]:
    cv2.imshow("Virtual Paint App", frame)
    cv2.imshow("Mask", mask)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


In [ ]:
cap.release()
cv2.destroyAllWindows()


In [3]:
import cv2
import numpy as np

# Blue color HSV range
lower = np.array([100, 150, 0])
upper = np.array([140, 255, 255])

# Stores drawing points
points = []

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower, upper)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    center = None

    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        if area > 500:
            (x, y, w, h) = cv2.boundingRect(cnt)
            center = (x + w//2, y + h//2)
            points.append(center)

    # Draw
    for i in range(1, len(points)):
        if points[i-1] is None or points[i] is None:
            continue
        cv2.line(frame, points[i-1], points[i], (255, 0, 0), 5)

    cv2.imshow("Virtual Paint App", frame)
    cv2.imshow("Mask", mask)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [5]:
import cv2
import numpy as np

# Pink color HSV range
lower = np.array([145, 80, 80])   # lower pink
upper = np.array([175, 255, 255]) # upper pink


points = []

cap = cv2.VideoCapture(0)
cap.set(3, 640)  # width
cap.set(4, 480)  # height


while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Reduce noise
    blur = cv2.GaussianBlur(frame, (7, 7), 0)

    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower, upper)

    # Morphology — removes noise, makes detection accurate
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    center = None

    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)

        if area > 800:  # more strict for accuracy
            M = cv2.moments(cnt)

            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                center = (cx, cy)
                points.append(center)
            else:
                points.append(None)
        else:
            points.append(None)
    else:
        points.append(None)

    # Draw smooth lines
    for i in range(1, len(points)):
        if points[i - 1] is None or points[i] is None:
            continue
        cv2.line(frame, points[i - 1], points[i], (255, 50, 0), 6)

    cv2.imshow("Virtual Paint App", frame)
    cv2.imshow("Mask", mask)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break
    elif key == ord('c'):  # Clear screen
        points = []

cap.release()
cv2.destroyAllWindows()


 # FULL ADVANCED VIRTUAL PAINT APP (with Features)

In [13]:
import cv2
import numpy as np

# -------------------------------
# HSV color ranges
# -------------------------------
color_ranges = {
    "pink":  ([145, 80, 80], [175, 255, 255]),
    "blue":  ([100, 150, 50], [140, 255, 255]),
    "red1":  ([0, 120, 70], [10, 255, 255]),
    "red2":  ([170, 120, 70], [180, 255, 255]),
    "green": ([40, 70, 70], [80, 255, 255]),
    "yellow": ([20, 100, 100], [35, 255, 255])
}

# Default drawing color
draw_color = (255, 0, 0)  # pink (BGR)

brush_thickness = 6
eraser_mode = False

# Canvas to draw on
canvas = np.zeros((480, 640, 3), dtype=np.uint8)

cap = cv2.VideoCapture(0)


def detect_color(hsv, lower, upper):
    lower = np.array(lower)
    upper = np.array(upper)
    mask = cv2.inRange(hsv, lower, upper)
    return mask
    

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    blur = cv2.GaussianBlur(frame, (7, 7), 0)
    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)

    # -------------------------------------
    # Detect only PINK pointer for drawing
    # -------------------------------------
    lower_p, upper_p = color_ranges["yellow"]
    mask = detect_color(hsv, lower_p, upper_p)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    center = None

    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        if area > 800:
            M = cv2.moments(cnt)
            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                center = (cx, cy)

                # -------------------------------
                # DRAWING ON CANVAS
                # -------------------------------
                if eraser_mode:
                    cv2.circle(canvas, center, brush_thickness + 10, (0, 0, 0), -1)
                else:
                    cv2.circle(canvas, center, brush_thickness, draw_color, -1)

    # -------------------------------------
    # Overlay canvas on frame
    # -------------------------------------
    final = cv2.add(frame, canvas)

    # -------------------------------------
    # Show UI Text
    # -------------------------------------
    cv2.putText(final, "Press C: Clear | E: Eraser | P: Pen | S: Save | +/- : Thickness",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    cv2.imshow("Virtual Paint (Advanced)", final)
    cv2.imshow("Mask", mask)

    key = cv2.waitKey(1) & 0xFF

    # Keyboard controls
    if key == ord('q'):  
        break
    elif key == ord('c'):
        canvas = np.zeros((480, 640, 3), dtype=np.uint8)
    elif key == ord('e'):  
        eraser_mode = True
    elif key == ord('p'):
        eraser_mode = False
        draw_color = (255, 255, 255)   # white color
    elif key == ord('+'):
        brush_thickness += 2
    elif key == ord('-'):
        brush_thickness = max(2, brush_thickness - 2)
    elif key == ord('s'):
        cv2.imwrite("drawing.png", canvas)
        print("Saved as drawing.png")

cap.release()
cv2.destroyAllWindows()


# 🔵 Lesson 13 — Barcode & QR Code Detection (Beginner–Friendly & Practical)

In [1]:
import cv2
from pyzbar import pyzbar


In [2]:
def decode(frame):
    decoded_objects = pyzbar.decode(frame)
    for obj in decoded_objects:
        # Draw bounding box
        (x, y, w, h) = obj.rect
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        # Display decoded text
        text = obj.data.decode("utf-8")
        cv2.putText(frame, text, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    return frame, decoded_objects


In [4]:
img = cv2.imread("C:/Users/Shyam/OneDrive/Desktop/Python Project/CodingWSagar/esewa_qr.png")
result, codes = decode(img)

print("Detected:", codes)

cv2.imshow("QR / Barcode", result)
cv2.waitKey(0)
cv2.destroyAllWindows()


Detected: [Decoded(data=b'fonepay://pay?pa=9701042653&pn=Aatish&cu=NPR&tn=testing', type='QRCODE', rect=Rect(left=40, top=40, width=330, height=330), polygon=[Point(x=40, y=40), Point(x=40, y=369), Point(x=370, y=370), Point(x=369, y=40)], quality=1, orientation='UP')]


In [6]:
import cv2
from pyzbar import pyzbar

def decode(frame):
    decoded = pyzbar.decode(frame)
    for obj in decoded:
        (x, y, w, h) = obj.rect

        # Draw box
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 3)

        text = obj.data.decode("utf-8")
        cv2.putText(frame, text, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    return frame, decoded

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    frame, decoded = decode(frame)

    cv2.imshow("QR / Barcode Scanner", frame)

    if decoded:
        print("Detected:", decoded[0].data.decode("utf-8"))

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
